# 2. Train and evaluate a seizure forecast model

Run **01_prepare_data.ipynb** first. This notebook starts from its saved handoff and does not repeat signal preprocessing. Run cells top to bottom with **Shift+Enter**; after changing parameters, restart the kernel and run from the top.

**Data flow:** saved feature rows → disjoint patient partitions → training-only grouped model comparison → validation-only alarm threshold → one held-out test evaluation → saved model and reports.

| Data role | What it does |
|---|---|
| Training patients | Learn cleanup/scaling/feature selection and model weights; compare parameter choices using patient-grouped folds |
| Validation patients | Choose the alarm threshold after the winning model is fixed |
| Test patients | Estimate performance once after those choices are fixed |

The same person's windows never appear in two roles. Shuffling adjacent windows into random train/test sets would leak patient information. Model probabilities are scores and are not automatically calibrated clinical risks. Results describe eligible forecast windows in this research dataset.

The default is a small, understandable logistic-regression experiment. Optional tree models are shown in the parameters. The project's LSTM path remains available through the existing CLI for later work; this notebook teaches the tabular feature workflow.


## 1. Load tools and find the project

**Input:** the project and Python environment. **Output:** project/interpreter paths. Use the same kernel as notebook 1; its first cell includes installation instructions.


In [ ]:
import json
import re
import sys
import time
import traceback
from contextlib import contextmanager, redirect_stdout, redirect_stderr
from datetime import datetime, timezone
from pathlib import Path

# Works when Jupyter starts in the project root or a subfolder of this project.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "utils" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the seizure_prediction project.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython import get_ipython
from IPython.display import display
from src.utils.config import load_config
from src.preprocessing.stream_processor import digest, save_json

if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
pd.set_option("display.max_columns", 12)
print("Project:", ROOT)
print("Python:", sys.executable)
print("Tip: if imports fail, select the project's .venv kernel.")


## 2. Choose training parameters

**Input:** your dataset name and model settings. **Output:** variables for this experiment. You do not re-enter sensor/window/label parameters here: they come from notebook 1.

`PATIENT_SPLITS = None` assigns prepared eligible patients in sorted ID order: approximately 70% training, 15% validation, and the rest test, while reserving at least one patient for validation/test. This is a transparent starting assignment, independent of model scores. With the current ten patients it produces **7 train / 1 validation / 2 test**. Review the counts in section 4; folder count alone does not ensure usable classes/events.

For a planned study, enter explicit participant lists. A failed readiness check stops; it does not quietly change the horizon, event minimums or splits. Small pilot changes should be recorded as a separate experiment.

Each model grid value is a list: `[1.0]` tries one setting; `[0.1, 1.0, 10.0]` compares three. `FEATURE_K` caps how many features are selected per training fold. A single logistic model is the fastest starting point. The commented tree definitions can be added to `MODEL_GRIDS` when you are ready.


In [ ]:
DATASET_NAME = "seizeit2_example"  # Must match notebook 1.
PATIENT_SPLITS = None
# Example explicit assignment for the current ten-patient cohort:
# PATIENT_SPLITS = {
#     "train": ["sub-001", "sub-002", "sub-003", "sub-004", "sub-005", "sub-006", "sub-007"],
#     "validation": ["sub-008"],
#     "test": ["sub-009", "sub-010"],
# }
SEED = 42
INNER_FOLDS = 3                   # Each inner validation fold holds out whole training patients.
MIN_EVENTS_PER_PARTITION = 2      # Count unique forecastable seizures, not positive windows.
FEATURE_K = 50                    # Capped at the number of surviving features.
MAX_FEATURE_MATRIX_GIB = 4        # Guard for the dense matrix; fitting needs additional RAM.
N_JOBS = 1                       # Start with one process to limit memory use.
NATIVE_THREADS = 1
MODEL_GRIDS = {
    "logistic": {"C": [0.1, 1.0], "max_iter": [2000], "class_weight": ["balanced"]},
    # "random_forest": {"n_estimators": [100], "max_depth": [8],
    #                   "min_samples_leaf": [2], "class_weight": ["balanced"]},
    # "hist_gradient_boosting": {"max_iter": [100], "max_leaf_nodes": [15],
    #                            "learning_rate": [0.1], "class_weight": ["balanced"]},
}
ALARM_THRESHOLDS = [0.1, 0.3, 0.5, 0.7, 0.9, 0.99, 1.0]
CONSECUTIVE_WINDOWS = 3           # Number of consecutive high-score decisions needed for an alarm.
REFRACTORY_SEC = 2400             # Minimum time between alarms.
MAX_FALSE_ALARMS_PER_ELIGIBLE_DAY = 2
BOOTSTRAP_REPLICATES = 0          # Off for this small example; 1000 enables patient-cluster intervals.
DEBUG = True


## 3. Read the preparation handoff and save a training experiment

**Input:** the handoff from notebook 1. **Output:** the exact prepared config/manifest and a new experiment directory. Hash checks detect edited settings or a changed manifest. No model is fitted.

The saved cohort stays fixed even if raw participant folders are added later. Missing/no-eligible participants are printed so you can review them. Training outputs are isolated by timestamp; rerunning creates another experiment rather than replacing a model.


In [ ]:
if not re.fullmatch(r"[A-Za-z0-9_-]+", DATASET_NAME):
    raise ValueError("DATASET_NAME must contain only letters, numbers, underscores or hyphens.")
WORKSPACE = ROOT / "audit" / "runs" / "notebooks" / DATASET_NAME
HANDOFF_FILE = WORKSPACE / "handoff.json"
if not HANDOFF_FILE.exists():
    raise FileNotFoundError(f"No completed handoff at {HANDOFF_FILE}. Finish notebook 1 first.")
handoff = json.loads(HANDOFF_FILE.read_text(encoding="utf-8"))
if handoff.get("schema_version") != 1 or handoff.get("status") != "prepared":
    raise ValueError("Unsupported or incomplete handoff. Rerun notebook 1.")
config = load_config(handoff["config_file"])
if digest(config) != handoff["config_digest"]:
    raise ValueError("Preparation settings changed after the handoff. Rerun notebook 1.")
manifest = json.loads(Path(handoff["feature_manifest"]).read_text(encoding="utf-8"))
if digest(manifest) != handoff["manifest_digest"]:
    raise ValueError("Feature manifest changed after the handoff. Rerun notebook 1.")
subjects = sorted(handoff["subjects"])
if len(subjects) < INNER_FOLDS + 2:
    raise ValueError(f"Need at least {INNER_FOLDS + 2} eligible patients for {INNER_FOLDS} folds plus validation/test; found {len(subjects)}.")
if PATIENT_SPLITS is None:
    n_train = max(INNER_FOLDS, min(int(len(subjects) * 0.70), len(subjects) - 2))
    n_validation = max(1, min(int(len(subjects) * 0.15), len(subjects) - n_train - 1))
    splits = {"train": subjects[:n_train], "validation": subjects[n_train:n_train + n_validation],
              "test": subjects[n_train + n_validation:]}
else:
    splits = PATIENT_SPLITS
if set(splits) != {"train", "validation", "test"}:
    raise ValueError("PATIENT_SPLITS needs exactly train, validation, and test.")
if any(not isinstance(names, list) or len(names) != len(set(names)) for names in splits.values()):
    raise ValueError("Each partition must be a list of unique participant IDs.")

config["training"].update(seed=SEED, splits=splits, inner_folds=INNER_FOLDS,
                          min_positive_events_per_partition=MIN_EVENTS_PER_PARTITION,
                          max_feature_matrix_gib=MAX_FEATURE_MATRIX_GIB, n_jobs=N_JOBS,
                          native_threads=NATIVE_THREADS, models=MODEL_GRIDS, selection_grid={"k": [FEATURE_K]})
config["training"]["selection"]["k"] = FEATURE_K
config["training"]["lstm"]["enabled"] = False
config["training"]["alarms"].update(thresholds=ALARM_THRESHOLDS, consecutive_windows=CONSECUTIVE_WINDOWS,
                                    refractory_sec=REFRACTORY_SEC, max_false_alarms_per_day=MAX_FALSE_ALARMS_PER_ELIGIBLE_DAY)
config["validation"].update(bootstrap_replicates=BOOTSTRAP_REPLICATES, plots=True, seed=SEED)
# Show all prepared patients to the split checks; never silently filter unassigned people.
config["qualification"]["exclude_unassigned_subjects"] = False
config["training"]["output_dir"] = str(WORKSPACE / "training")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = WORKSPACE / "training" / f"{stamp}_{digest(config)}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
training_config_file = RUN_DIR / "config.yaml"
training_config_file.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
config = load_config(training_config_file)
save_json(RUN_DIR / "config.json", config)
save_json(RUN_DIR / "preparation_handoff.json", handoff)
LOG_FILE = RUN_DIR / "training.log"
print("Experiment:", RUN_DIR)
print("Prepared but no eligible rows:", sorted(set(handoff["selected_subjects"]) - set(subjects)) or "none")
display(pd.DataFrame([{"partition": name, "participants": ids} for name, ids in splits.items()]))


## 4. Check patient splits and forecastable events before fitting

**Input:** saved window metadata; no large feature matrix is needed yet. **Output:** patient/partition counts, the exact grouped folds, `readiness.json`, and a pass/fail message.

Every partition needs background and positive windows plus the chosen minimum number of unique seizure episodes. Every inner fold must contain both classes. An event is counted only when at least one eligible forecast decision can target it. These checks use annotation availability, not model/test scores.

If readiness fails, inspect the printed counts. Correct a mistaken split, add eligible data, or deliberately design a separately named pilot with different requirements. Repeatedly trying splits to improve held-out performance would invalidate the evaluation.


In [ ]:
# Tee sends ordinary printed output to both the notebook and a text log.
class Tee:
    def __init__(self, screen, log):
        self.screen, self.log = screen, log

    def write(self, text):
        self.screen.write(text)
        self.log.write(text)
        return len(text)

    def flush(self):
        self.screen.flush()
        self.log.flush()


@contextmanager
def logged_step(name):
    start = time.perf_counter()
    with LOG_FILE.open("a", encoding="utf-8") as log:
        with redirect_stdout(Tee(sys.stdout, log)), redirect_stderr(Tee(sys.stderr, log)):
            print(f"\nSTART {name} | {datetime.now(timezone.utc).isoformat()}", flush=True)
            try:
                yield
            except Exception:
                traceback.print_exc()  # Keep the full error in the log, then stop the cell.
                raise
            finally:
                print(f"END {name} | elapsed {time.perf_counter() - start:.1f} seconds", flush=True)


print("Detailed progress and errors:", LOG_FILE)


from src.training.data import load_prepared_metadata, load_feature_data
from src.training.readiness import report as readiness_report
from src.training.splits import preflight

with logged_step("Check labels, patient partitions and grouped folds"):
    metadata = load_prepared_metadata(config)
    readiness = readiness_report(config, metadata)
    save_json(RUN_DIR / "readiness.json", readiness)
    print("READY:", readiness["ready"], "| Forecast interval (minutes):", readiness["forecast_minutes"])
    for issue in readiness["issues"]:
        print("NOT READY:", issue)
display(pd.DataFrame(readiness["patients"]))
display(pd.DataFrame.from_dict(readiness["partitions"], orient="index"))
if DEBUG:
    display(pd.DataFrame(readiness["inner_folds"]))
if not readiness["ready"]:
    raise ValueError("Training stopped before fitting. " + "; ".join(readiness["issues"]))


## 5. Load the feature matrix and verify its rows

**Input:** locally generated feature files. **Output:** `X` (rows = eligible windows; columns = signal features), `meta` (matching labels/patient IDs/times), event tables and partition row indices.

Loading has an estimated memory guard. The actual matrix plus model-selection copies can use more memory than `X` alone. For a larger cohort, prepare a manageable subset or use a higher-memory machine; discovering 100 patients does not make this an out-of-core trainer.

Missing feature values remain `NaN` here. The training pipeline learns filling/scaling on training data only. The debug preview uses training patients only.


In [ ]:
with logged_step("Load eligible feature rows"):
    X, meta, events, entries = load_feature_data(config)
    indices, folds = preflight(config, X, meta)
    print(f"X: {X.shape[0]:,} rows × {X.shape[1]} features")
    print(f"Dense table memory: {X.memory_usage(deep=True).sum() / 1024**3:.3f} GiB; fitting needs more.")
    print("Partition rows:", {name: len(rows) for name, rows in indices.items()})
    if not X.index.equals(meta.index) or len(X) != len(meta):
        raise ValueError("Feature and label rows do not align.")
    if not meta.label.isin([0, 1]).all():
        raise ValueError("Excluded/unknown labels entered the training table.")
if DEBUG:
    display(X.iloc[indices["train"][:5], :8])
    display(meta.iloc[indices["train"][:5]][["subject", "recording_id", "prediction_time_sec", "label"]])
    display(X.iloc[indices["train"]].isna().mean().sort_values(ascending=False).head(10).rename("training missing fraction"))


## 6. See exactly what each candidate learns

**Input:** model grids and selection settings. **Output:** a candidate table and an unfitted pipeline.

Each pipeline first removes unusable/constant/redundant features, learns median filling and scaling, then selects up to `FEATURE_K` features, then fits its classifier. All these learned steps are cloned and fitted inside each training fold. Feature ranking is never performed on the full dataset first.

**Average precision** compares how well a model ranks rare positive windows; higher is better. Its chance baseline depends on positive-window prevalence. The scores in the next cell are from training-patient folds, not the final test patients.


In [ ]:
from sklearn.model_selection import GridSearchCV, ParameterGrid
from src.training.registry import candidates
from threadpoolctl import threadpool_limits

candidate_models = candidates(config)
display(pd.DataFrame([
    {"model": name, "parameter_combinations": len(ParameterGrid(grid)),
     "folds": len(folds), "fits_plus_refit": len(ParameterGrid(grid)) * len(folds) + 1}
    for name, (_, grid) in candidate_models.items()
]))
display(next(iter(candidate_models.values()))[0])
print("Feature cleanup settings:", config["training"]["selection"])


## 7. Fit and compare models using training patients only

**Input:** training rows and grouped inner folds. **Output:** a comparison table, full CV results per candidate and the best pipeline refitted on all training patients. Each completed fit is printed and logged. Validation/test predictions are not used here.

Changing `C` changes logistic regularization; the fold scores choose between the listed values. This visible loop replaces the dashboard's hidden training job. Interrupting here leaves an incomplete experiment folder without a completion marker.


In [ ]:
comparison_rows = []
best_score, best_model, best_name = -np.inf, None, None
train_rows = indices["train"]
with logged_step("Training-only grouped model comparison"):
    with threadpool_limits(limits=NATIVE_THREADS):
        for name, (pipeline, grid) in candidate_models.items():
            print("Fitting:", name, flush=True)
            search = GridSearchCV(pipeline, grid, cv=folds, scoring="average_precision",
                                  n_jobs=N_JOBS, refit=True, error_score="raise", verbose=2)
            search.fit(X.iloc[train_rows], meta.label.iloc[train_rows])
            pd.DataFrame(search.cv_results_).to_csv(RUN_DIR / f"{name}_inner_cv.csv", index=False)
            comparison_rows.append({"model": name, "inner_cv_average_precision": float(search.best_score_),
                                    "parameters": search.best_params_})
            if search.best_score_ > best_score:
                best_score, best_model, best_name = search.best_score_, search.best_estimator_, name
if best_model is None:
    raise RuntimeError("No model completed fitting. Inspect training.log.")
save_json(RUN_DIR / "model_comparison.json", comparison_rows)
display(pd.DataFrame(comparison_rows).sort_values("inner_cv_average_precision", ascending=False))
selected_features = best_model.named_steps["features"].get_feature_names_out().tolist()
print("Chosen model:", best_name, "| Selected features:", len(selected_features))
display(pd.Series(selected_features[:15], name="first selected features"))


## 8. Choose the alarm threshold on validation patients

**Input:** the fixed model's validation scores and the alarm rules. **Output:** a threshold comparison table and the chosen threshold. The model is not refitted here.

A score above the threshold contributes to an alarm after `CONSECUTIVE_WINDOWS` consecutive decisions; the refractory period limits repeat alarms. The selection maximizes eligible-event sensitivity subject to your validation false-alarm budget. With a 60-second stride, three consecutive windows means three decisions spaced a minute apart.

A threshold of `1.0` often produces no alarms, but a model can score exactly 1.0. If no candidate meets the budget, this cell stops before test prediction. No alarms/zero sensitivity can be a valid result for a weak model; it does not demonstrate useful forecasting.


In [ ]:
from src.models.evaluation import choose_threshold, window_metrics, alarm_metrics

validation_rows = indices["validation"]
validation_meta = meta.iloc[validation_rows].copy()
with logged_step("Select threshold using validation patients"):
    with threadpool_limits(limits=NATIVE_THREADS):
        validation_probabilities = best_model.predict_proba(X.iloc[validation_rows])[:, 1]
    chosen_alarm, threshold_results = choose_threshold(validation_meta, validation_probabilities, config, events)
    threshold = chosen_alarm["threshold"]
    save_json(RUN_DIR / "validation_thresholds.json", threshold_results)
    print("Chosen threshold:", threshold)
display(pd.DataFrame(threshold_results)[["threshold", "event_sensitivity", "false_alarms_per_eligible_day", "alarms"]])
# Baseline is learned from training labels only, before evaluating held-out predictions.
reference_probability = float(meta.label.iloc[train_rows].mean())
validation_predictions = validation_meta.assign(probability=validation_probabilities)


## 9. Evaluate the fixed choices on test patients

**Input:** the chosen model/threshold and held-out test rows. **Output:** validation/test metric tables, saved predictions, per-patient/event reports and diagnostic plots. This is the first test prediction.

Do not tune parameters after looking at test results and then claim the same test is untouched. Run this cell once per completed experiment. A guard prevents accidental repeated evaluation in the same kernel. Restarting and retraining is a new experiment, not a new independent test cohort.

Window precision/recall and average precision describe window classification. Event sensitivity describes the fraction of forecastable seizures warned about. False alarms are normalized by **eligible observation support**, not all recording time; sparse stride and exclusions limit that interpretation. Confidence intervals are disabled by default for the tiny held-out cohort.


In [ ]:
from src.validation.report import write_report

if globals().get("test_evaluated", False):
    raise RuntimeError("Test already evaluated in this kernel. Inspect the saved results instead of rerunning this cell.")
test_rows = indices["test"]
with logged_step("Evaluate fixed model on held-out test patients"):
    with threadpool_limits(limits=NATIVE_THREADS):
        test_probabilities = best_model.predict_proba(X.iloc[test_rows])[:, 1]
    test_evaluated = True  # Set immediately so a later report error cannot silently reopen the test.
    test_predictions = meta.iloc[test_rows].copy().assign(probability=test_probabilities)
    report_results = {}
    evaluation = {"selected_model": best_name, "threshold": threshold,
                  "probability_note": "Class-weighted probabilities are not independently calibrated"}
    for partition, predictions in [("validation", validation_predictions), ("test", test_predictions)]:
        predictions.to_csv(RUN_DIR / f"{partition}_predictions.csv", index=False)
        report_results[partition] = write_report(predictions, config, events, threshold, reference_probability,
                                                  RUN_DIR / "validation_reports" / partition, partition)
        evaluation[partition] = report_results[partition]["window"]
        evaluation[f"{partition}_alarms"] = report_results[partition]["alarm"]
    save_json(RUN_DIR / "evaluation.json", evaluation)
display(pd.DataFrame({name: result["window"] for name, result in report_results.items()}).loc[
    ["windows", "positive_windows", "prevalence", "average_precision", "roc_auc", "precision", "recall", "brier_score"]])
display(pd.DataFrame({name: result["alarm"] for name, result in report_results.items()}).loc[
    ["eligible_events", "warned_events", "event_sensitivity", "false_alarms_per_eligible_day", "alarms"]])


## 10. Save the fitted pipeline and verify that it reloads

**Input:** the fitted pipeline, fixed threshold, feature names and predictions. **Output:** `model.joblib`, configuration/split/version snapshots and a completion marker. A reload check confirms the saved model reproduces its validation predictions.

Save the whole pipeline, not just classifier weights: its filling/scaling/feature selection are needed to make matching predictions. Load only your own trusted joblib/pickle artifacts. New inference rows must come from the same feature preparation and named-column convention; identifiers and annotations do not belong in `X`.

The existing `src.commands.validate_model` CLI can regenerate reports from this experiment folder without fitting again. `complete.json` appears only after the saved-model verification succeeds.


In [ ]:
import importlib.metadata
import joblib

with logged_step("Save and reload the complete fitted pipeline"):
    save_json(RUN_DIR / "feature_manifest.json", entries)
    save_json(RUN_DIR / "splits.json", config["training"]["splits"])
    save_json(RUN_DIR / "inner_folds.json", readiness["inner_folds"])
    save_json(RUN_DIR / "selected_features.json", selected_features)
    save_json(RUN_DIR / "selected_channel_groups.json", sorted({"::".join(name.split("::")[:2]) for name in selected_features}))
    save_json(RUN_DIR / "validation_reference.json", {"training_prevalence": reference_probability,
              "training_windows": len(train_rows), "source": "unweighted eligible training labels only"})
    save_json(RUN_DIR / "versions.json", {package: importlib.metadata.version(package)
              for package in ["numpy", "scipy", "pandas", "scikit-learn", "mne", "joblib"]})
    artifact_file = RUN_DIR / "model.joblib"
    joblib.dump({"model": best_model, "threshold": threshold, "config": config,
                 "selected_features": selected_features}, artifact_file)
    reloaded = joblib.load(artifact_file)
    with threadpool_limits(limits=NATIVE_THREADS):
        reloaded_probabilities = reloaded["model"].predict_proba(X.iloc[validation_rows])[:, 1]
    np.testing.assert_allclose(reloaded_probabilities, validation_probabilities, rtol=1e-12, atol=1e-12)
    save_json(RUN_DIR / "complete.json", {"status": "complete", "selected_model": best_name,
              "saved_model_reload_predictions_match": True})
print("TRAINING COMPLETE")
print("Model:", artifact_file)
print("Reports:", RUN_DIR / "validation_reports")
print("Log:", LOG_FILE)


## 11. Inspect the saved reports without fitting again

**Input:** completed report files. **Output:** a per-patient table and saved PR/ROC/calibration figure. PR = precision–recall; ROC compares true/false positive rates; calibration compares score bins with observed positive fractions.

After a plotting/reporting problem, inspect the files before restarting training. A failed experiment has no top-level `complete.json`. The last completed fit and any CSV files remain available for diagnosis.


In [ ]:
from IPython.display import Image

display(pd.read_csv(RUN_DIR / "validation_reports" / "test" / "per_patient.csv").iloc[:, :10])
display(Image(filename=str(RUN_DIR / "validation_reports" / "test" / "curves.png")))
display(pd.DataFrame([
    {"file": "model.joblib", "contains": "fitted cleanup, selected features, classifier and threshold"},
    {"file": "config.yaml / preparation_handoff.json", "contains": "training settings and exact prepared dataset"},
    {"file": "readiness.json / splits.json", "contains": "class/event counts and patient assignments"},
    {"file": "*_inner_cv.csv / model_comparison.json", "contains": "training-only model comparisons"},
    {"file": "validation_thresholds.json", "contains": "validation-only threshold comparison"},
    {"file": "*_predictions.csv", "contains": "fixed held-out probabilities and labels"},
    {"file": "validation_reports/", "contains": "metrics, per-patient/event/alarm tables and plots"},
    {"file": "training.log / complete.json", "contains": "progress/errors and successful completion status"},
]))


## Debugging and the next experiment

| Symptom | What to do |
|---|---|
| Missing handoff / edited snapshot | Complete notebook 1 with the same `DATASET_NAME` |
| A partition lacks a class/event | Read the patient counts and correct the study split/data |
| A grouped fold lacks positives | Add eligible training patients/events or deliberately redesign the pilot |
| Stale features | Return to notebook 1; changing labels/sensors/features requires preparation |
| Feature-memory guard | Use a smaller prepared cohort or more RAM; CV creates additional copies |
| No usable features | Inspect training missingness/constant signals and the source windows |
| No feasible threshold | Inspect validation threshold results and the chosen alarm rules; test remains unpredicted |
| Kernel interrupted / fit error | Read `training.log`; rerun from the top after correcting the cause |

Temporary `%debug` after an error opens the interactive debugger (`p variable`, `q` to exit). `display(X.shape)`, `display(meta.head())` and the saved readiness/CV tables are useful first checks. Do not replace failed assertions with unconditional success.

Change model grids or selection size here to reuse the prepared dataset. Change the forecast target, sensor selection or signal windows in notebook 1. More participants require preparation again, then deliberate patient assignment. Advanced LSTM experiments remain described in `documentation/TRAINING_PIPELINE.md`.
